# 🚀 Gemini Enterprise Starter Pack — Marketing

**5 production-ready use cases for Marketing teams using the Gemini API**

| Use Case | Description |
|----------|-------------|
| UC-MKT-01 | Multi-channel content from a single brief |
| UC-MKT-02 | Market research & competitive intelligence summarization |
| UC-MKT-03 | A/B testing variant generation |
| UC-MKT-04 | Creative brief structuring from raw notes |
| UC-MKT-05 | Persona creation from customer data |

---
**Author:** Rania Mehria — Outcome Customer Engineer, Google Cloud  
**Model:** `gemini-2.0-flash` (fast) or `gemini-1.5-pro` (for long documents)  
**Docs:** https://ai.google.dev/gemini-api/docs


## ⚙️ Setup & Authentication

In [ ]:
# Install the Gemini SDK
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json

# ── Authentication ──────────────────────────────────────────────────────────
# Option A: Gemini API Key (for prototyping)
# Store your key in Colab Secrets (🔑 icon) as GEMINI_API_KEY
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    print('✅ Authenticated via API Key')
except Exception:
    print('⚠️  No Colab secret found — add GEMINI_API_KEY in the Secrets panel (🔑)')
    print('   Or replace with: genai.configure(api_key="YOUR_KEY")')

# Option B: Vertex AI (for enterprise / Google Cloud projects)
# Uncomment and use this instead for production:
# import vertexai
# vertexai.init(project='YOUR_PROJECT_ID', location='europe-west1')
# from vertexai.generative_models import GenerativeModel

# ── Model selection ─────────────────────────────────────────────────────────
# gemini-2.0-flash  → fast, great for most use cases
# gemini-1.5-pro    → best for long documents (up to 2M tokens)
MODEL_NAME = 'gemini-2.0-flash'
model = genai.GenerativeModel(MODEL_NAME)
print(f'✅ Model loaded: {MODEL_NAME}')

---
## UC-MKT-01 — Multi-Channel Content from a Single Brief

In [ ]:
def generate_multichannel_content(
    company: str,
    product: str,
    target_audience: str,
    key_benefit: str,
    brand_tone: str,
    cta: str
) -> dict:
    """
    Generate LinkedIn post, nurturing email, Google Ads copy,
    and blog introduction from a single campaign brief.
    Returns a dict with each format as a key.
    """
    prompt = f"""You are a B2B content marketing expert.

Campaign brief:
- Company: {company}
- Product / Service: {product}
- Target audience: {target_audience}
- Key benefit: {key_benefit}
- Brand tone: {brand_tone}
- Call to action: {cta}

Generate all 4 formats below. Use === FORMAT NAME === as separator.

=== LINKEDIN POST ===
(max 250 words, strong 2-line hook, 3 paragraphs, 3–5 hashtags)

=== NURTURING EMAIL ===
(subject line on first line, then body, 150 words, use [FIRST_NAME] for personalization)

=== GOOGLE ADS ===
(3 headlines max 30 chars each, then 2 descriptions max 90 chars each)

=== BLOG INTRODUCTION ===
(200 words, editorial angle, rhetorical question, promise of the article)
"""

    response = model.generate_content(prompt)
    raw = response.text

    # Parse into structured dict
    sections = {}
    keys = ['LINKEDIN POST', 'NURTURING EMAIL', 'GOOGLE ADS', 'BLOG INTRODUCTION']
    for i, key in enumerate(keys):
        start = raw.find(f'=== {key} ===')
        if start == -1:
            continue
        start += len(f'=== {key} ===')
        end = raw.find('===', start) if i < len(keys) - 1 else len(raw)
        sections[key.lower().replace(' ', '_')] = raw[start:end].strip()

    return sections


# ── Example ──────────────────────────────────────────────────────────────────
result = generate_multichannel_content(
    company='Acme Corp',
    product='AI-powered HR platform',
    target_audience='HR Directors at companies with 200–2000 employees',
    key_benefit='Cut time-to-hire by 40% with automated CV screening',
    brand_tone='Professional, data-driven, no fluff',
    cta='Book a free 30-min demo'
)

for format_name, content in result.items():
    print(f'\n{'='*60}')
    print(f'📄 {format_name.upper().replace("_", " ")}')
    print('='*60)
    print(content)

---
## UC-MKT-02 — Market Research Summarization (PDF support)

In [ ]:
import pathlib

def summarize_market_report(
    report_text: str,
    company_name: str,
    sector: str,
    competitor_1: str = '',
    competitor_2: str = ''
) -> str:
    """
    Summarize a market research report and extract actionable insights.
    Pass report_text as plain text (extracted from PDF).
    For PDF upload, use the File API version below.
    """
    comp_section = ''
    if competitor_1 or competitor_2:
        comp_section = f"""
4. COMPETITIVE POSITIONING
   How are {competitor_1} and {competitor_2} positioned according to this report?
"""

    prompt = f"""You are a senior market analyst specializing in {sector}.

REPORT CONTENT:
{report_text[:50000]}  # Truncate for safety; use gemini-1.5-pro for full docs

Analyze this report and provide:

1. EXECUTIVE SUMMARY
   5 bullet points max — the most important trends only

2. OPPORTUNITIES FOR {company_name}
   3 concrete, immediately actionable opportunities

3. THREATS & WEAK SIGNALS
   3 points to watch closely
{comp_section}
5. MARKETING RECOMMENDATIONS
   3 priority actions for the Marketing team in the next 90 days

Format: structured headers, concise bullets, add a key quote per section when available.
"""
    response = model.generate_content(prompt)
    return response.text


def summarize_report_from_pdf(pdf_path: str, company_name: str, sector: str) -> str:
    """
    Summarize directly from a PDF file using Gemini's File API.
    Best for large documents — uses gemini-1.5-pro for 2M token context.
    """
    pro_model = genai.GenerativeModel('gemini-1.5-pro')
    pdf_file = genai.upload_file(pdf_path, mime_type='application/pdf')
    print(f'✅ PDF uploaded: {pdf_file.name}')

    response = pro_model.generate_content([
        pdf_file,
        f'You are a senior market analyst for {sector}. '
        f'Summarize this report for {company_name}: '
        '1) Executive summary (5 bullets), 2) Top 3 opportunities, '
        '3) Top 3 threats, 4) 3 marketing recommendations for the next 90 days.'
    ])
    return response.text


# ── Example with text ────────────────────────────────────────────────────────
sample_text = """
The global enterprise AI market is projected to reach $XX billion by 2027.
Key trends include: increased adoption of generative AI in productivity tools,
growing emphasis on AI governance and explainability, and consolidation among
vendors offering integrated AI platforms rather than point solutions...
(replace with your actual report content)
"""

summary = summarize_market_report(
    report_text=sample_text,
    company_name='Acme Corp',
    sector='enterprise SaaS',
    competitor_1='Competitor A',
    competitor_2='Competitor B'
)
print(summary)

---
## UC-MKT-03 — A/B Testing Variant Generator

In [ ]:
def generate_ab_variants(
    company: str,
    offer: str,
    persona: str,
    goal: str,
    constraint: str = 'email subject max 50 characters'
) -> str:
    """
    Generate email subject lines, landing page headlines,
    and CTA variants for A/B testing — organized by psychological angle.
    """
    prompt = f"""You are an expert B2B copywriter specializing in conversion.

Context:
- Company: {company}
- Offer: {offer}
- Target persona: {persona}
- Goal: {goal}
- Constraint: {constraint}

Generate:

A) 5 EMAIL SUBJECT LINES (one per angle):
   [BENEFIT] Direct quantified benefit
   [CURIOSITY] Provocative question
   [URGENCY] Scarcity / time pressure
   [SOCIAL PROOF] Benchmark / peer reference
   [PAIN] Pain point recognition

B) 4 LANDING PAGE HEADLINES (H1, max 10 words, same angle variety)

C) 3 CTA BUTTON TEXTS (3–5 words):
   [RATIONAL] logic-driven
   [EMOTIONAL] feeling-driven
   [IMMEDIATE] benefit right now

For each, add a one-line note on the psychology behind the choice.
"""
    response = model.generate_content(prompt)
    return response.text


# ── Batch variant generation ──────────────────────────────────────────────────
def batch_generate_variants(campaigns: list[dict]) -> list[dict]:
    """
    Generate A/B variants for multiple campaigns at once.
    Input: list of campaign dicts with keys: company, offer, persona, goal
    """
    results = []
    for i, campaign in enumerate(campaigns):
        print(f'Generating variants for campaign {i+1}/{len(campaigns)}...')
        variants = generate_ab_variants(**campaign)
        results.append({'campaign': campaign, 'variants': variants})
    return results


# ── Example ──────────────────────────────────────────────────────────────────
result = generate_ab_variants(
    company='Acme Corp',
    offer='AI-powered contract review tool',
    persona='Legal Directors at large enterprises',
    goal='registration for a live product demo',
    constraint='email subject max 50 characters, no ALL CAPS'
)
print(result)

---
## UC-MKT-04 — Creative Brief Generator

In [ ]:
def generate_creative_brief(project_type: str, raw_notes: str) -> str:
    """
    Transform raw notes into a structured, agency-ready creative brief.
    """
    prompt = f"""You are a senior marketing director.

I need a structured creative brief for: {project_type}

Raw input (notes, emails, bullet points — everything I have):
{raw_notes}

Generate a complete, professional creative brief with:
1. CONTEXT & OBJECTIVES (market context + SMART goal + success KPIs)
2. TARGET AUDIENCE (primary persona + key buyer insight)
3. KEY MESSAGES (core promise + 3 proof points + tone of voice)
4. DELIVERABLES (formats + technical specs + legal constraints)
5. TIMELINE (approval phases + key milestones)
6. REFERENCE MATERIALS (inspiring examples to find + what to avoid)

Format: professional document, clear structure, ready to send to an external agency.
Then add a CREATIVE DIRECTOR REVIEW section: 3 weaknesses in this brief and how to fix them.
"""
    response = model.generate_content(prompt)
    return response.text


# ── Example ──────────────────────────────────────────────────────────────────
raw = """
We want to launch a LinkedIn campaign for our new product.
It's a document automation tool for finance teams.
We need at least 200 registrations for our upcoming webinar.
Budget is around 5k euros. Target: CFOs and finance directors in France.
Deadline: campaign live in 3 weeks.
Tone should be serious but not boring. We don't want stock photos.
"""

brief = generate_creative_brief(
    project_type='LinkedIn awareness + lead generation campaign',
    raw_notes=raw
)
print(brief)

---
## UC-MKT-05 — Persona Generator from Customer Data

In [ ]:
def generate_personas(
    raw_customer_data: str,
    product_name: str,
    num_personas: int = 3
) -> str:
    """
    Generate detailed, actionable marketing personas from raw customer data.
    Input can be: CRM exports, interview notes, NPS verbatims, sales notes.
    """
    prompt = f"""You are a B2B marketing strategist and buyer psychology expert.

Below is raw customer data (anonymized). Analyze it and generate {num_personas} personas.

CUSTOMER DATA:
{raw_customer_data}

For each persona, produce a structured card:

PERSONA [N] — [FICTIONAL NAME, realistic]

DEMOGRAPHIC PROFILE
- Job title | Industry | Company size | Tenure in role

PSYCHOGRAPHIC PROFILE
- Top 3 professional priorities
- Top 3 daily frustrations
- Personal definition of success
- Preferred information sources (LinkedIn? Newsletters? Analysts?)

RELATIONSHIP TO {product_name}
- Purchase trigger (what makes them start looking for a solution?)
- Typical objections (top 2)
- Decision criteria: rational vs emotional split
- Decision process: solo or committee?

COMMUNICATION STRATEGY
- Best channel to reach them
- Content type that resonates
- The one message that drives action
- What they absolutely don't want to hear

TYPICAL QUOTE
"[What this persona would say on a discovery call — realistic verbatim]"

---
After the personas, add a QUALIFICATION GRID:
10 discovery call questions to identify which persona a prospect belongs to.
"""
    response = model.generate_content(prompt)
    return response.text


# ── Example ──────────────────────────────────────────────────────────────────
sample_data = """
Customer interviews (anonymized):
- VP Marketing, 500-person SaaS company: 'I spend 40% of my time on content that should take 10 minutes'
- Marketing Director, financial services: 'Compliance reviews kill our campaign velocity'
- CMO, retail group: 'My team can write, they just need structure. They waste time on briefs.'

NPS verbatims (promoters):
- 'Finally a tool that gets B2B tone right'
- 'Cut our time-to-publish by 60%'

NPS verbatims (detractors):
- 'Output needs too much editing for our industry'
- 'Hard to get it to respect our brand guidelines consistently'
"""

personas = generate_personas(
    raw_customer_data=sample_data,
    product_name='Gemini Enterprise',
    num_personas=3
)
print(personas)

---
## 🔁 Bonus — Pipeline: Brief → All Formats → Google Sheets Export

Automate the full Marketing workflow and export results to Google Sheets.

In [ ]:
# Full pipeline: brief → multi-channel content → structured export
# Requires: pip install gspread google-auth

def run_full_marketing_pipeline(brief: dict) -> dict:
    """
    End-to-end pipeline:
    1. Generate multi-channel content from brief
    2. Generate A/B variants for email
    3. Return structured dict ready for Sheets export
    """
    print('🚀 Running full marketing pipeline...')

    # Step 1: Multi-channel content
    print('   Step 1/2: Generating multi-channel content...')
    content = generate_multichannel_content(**brief)

    # Step 2: A/B email variants
    print('   Step 2/2: Generating A/B variants...')
    variants = generate_ab_variants(
        company=brief['company'],
        offer=brief['product'],
        persona=brief['target_audience'],
        goal=f"Promote: {brief['key_benefit']}",
    )

    result = {
        'brief': brief,
        'linkedin_post': content.get('linkedin_post', ''),
        'email': content.get('nurturing_email', ''),
        'google_ads': content.get('google_ads', ''),
        'blog_intro': content.get('blog_introduction', ''),
        'ab_variants': variants,
    }

    print('✅ Pipeline complete!')
    return result


# ── Run the pipeline ──────────────────────────────────────────────────────────
brief = {
    'company': 'Acme Corp',
    'product': 'AI-powered contract automation platform',
    'target_audience': 'Legal Operations Directors at Fortune 500 companies',
    'key_benefit': 'Reduce contract review time by 70%',
    'brand_tone': 'Expert, data-driven, no fluff',
    'cta': 'Request a live demo'
}

pipeline_output = run_full_marketing_pipeline(brief)

# Print results
for key, value in pipeline_output.items():
    if key != 'brief':
        print(f'\n{"="*50}')
        print(f'📄 {key.upper()}')
        print('='*50)
        print(value[:500] + '...' if len(str(value)) > 500 else value)